<a href="https://colab.research.google.com/github/litlsun/practicum-summarization-bot/blob/develop/notebooks/ya_sum_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 YaSumBot - Суммаризация вебинаров

Бот для Telegram, который преобразует аудио/видео лекции в структурированный конспект с заданиями.

---

## Структура блокнота:

### 1. Загрузка файла с Яндекс.Диска
- Проверка ссылки (домен, доступ, папка/файл)
- Проверка формата (аудио/видео)
- Скачивание файла по публичной ссылке

### 2. Транскрибация (Whisper)
- Загрузка модели Whisper (medium)
- Преобразование аудио/видео в текст
- Сохранение расшифровки в TXT

### 3. Суммаризация (Alice AI LLM)
- Настройка клиента Yandex Cloud
- Инструкция для LLM (фильтрация, структура, задания)
- Генерация структурированного урока
- Создание 5 типов заданий с ответами
- Сохранение в Markdown и TXT

### 4. Telegram Bot
- Обработка команды /start
- Приём ссылок от пользователя
- Валидация ссылки
- Скачивание → транскрибация → суммаризация
- Отправка результатов (расшифровка + урок + задания)
- Очистка временных файлов
- Обработка ошибок

---

## Как использовать:


### Шаг 1: Добавьте секреты
Нажмите на иконку 🔑 **Secrets** (слева) и добавьте:
| Имя секрета | Описание | Где взять |
|-------------|----------|-----------|
| `YANDEX_API` | API ключ Yandex Cloud | [Консоль Yandex Cloud](https://console.cloud.yandex.ru) → API-ключи |
| `TG_BOT` | Токен Telegram бота | [@BotFather](https://t.me/BotFather) в Telegram |

### Шаг 2: Настройте Yandex Cloud
В главе **Суммаризация** найдите строку:
```python
YANDEX_CLOUD_FOLDER = ""  # вставьте ID папки из консоли Yandex.Cloud
```

### Шаг 3: Запуск
1. **Включите GPU**: Runtime → Change runtime type → T4 GPU → Save
2. **Запустите всё**: Runtime → Run all
3. **Идите в Telegram** → отправьте боту `/start` и ссылку на файл

# Загрузка файла по ссылке с Яндекс.Диска

In [ ]:
import requests
import os

In [ ]:
def is_yandex_file(url):
    """
    Проверяет, правильная ли ссылка на аудио- видео-файл Яндекс.Диска.

    Args:
        url (str): Публичная ссылка на файл Яндекс.Диска

    Returns:
        tuple: (is_valid, message, filename)
            - is_valid (bool): True если ссылка валидна
            - message (str): Сообщение для пользователя
            - filename (str или None): Имя файла или None при ошибке
    """
    # Проверка, что это ссылка Яндекс.Диска
    is_yandex_domain = (
        'yadi.sk' in url or
        'disk.yandex.ru' in url or
        'disk.360.yandex.ru' in url
    )
    if not is_yandex_domain:
        return (
            False,
            "Поддерживаются только ссылки на файлы с Яндекс.Диска!",
            None
            )

    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources"

    # Поддерживаемые форматы
    supported_audio = ['.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg', '.wma']
    supported_video = [
        '.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv', '.mpeg', '.mpg'
        ]
    supported_formats = supported_audio + supported_video

    try:
        response = requests.get(api_url, params={"public_key": url}, timeout=10)

        if response.status_code != 200:
            return (
                False,
                'Нет доступа! Проверьте, что ссылка публичная, '
                'кликните на иконку "Поделиться" > "Скопировать" на Яндекс.Диске',
                None
                )

        data = response.json()

        # Проверка на папку
        if data.get('type') != 'file':
            return (
                False,
                "Скорее всего, это ссылка на папку, а не на файл!",
                None
                )

        # Проверка формата
        filename = data.get('name', 'файл')
        file_ext = os.path.splitext(filename)[1].lower()

        if file_ext not in supported_formats:
            return False, "Я работаю только с аудио и видео!", None

        # Если всё хорошо
        return (
            True,
            f'✨ Супер! Файл "{filename}" в порядке! Я готов приступать к работе! 💪',
            filename
            )

    except Exception:
        return (
            False,
            'Что-то не так со ссылкой!',
            None
            )

In [ ]:
def download_yandex_file(url, save_path):
    """
    Скачивает файл с Яндекс.Диска по публичной ссылке.

    Args:
        url (str): Публичная ссылка на файл Яндекс.Диска
        save_path (str): Путь для сохранения файла

    Returns:
        bool: True в случае успеха
    """
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

    response = requests.get(
        api_url,
        params={"public_key": url}
    )
    response.raise_for_status()

    download_url = response.json()["href"]

    file_response = requests.get(download_url, stream=True)
    file_response.raise_for_status()

    with open(save_path, "wb") as f:
        for chunk in file_response.iter_content(chunk_size=8192):
            f.write(chunk)

    return True

# Транскрибация

In [ ]:
!pip install openai-whisper

In [ ]:
import whisper
import torch

# Загрузка модели Whisper для распознавания речи

# Определяем устройство: GPU если доступен, иначе CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
# Загружаем модель medium
model = whisper.load_model("medium", device=device)

In [ ]:
def transcribe(audio_path, save_path="transcription.txt"):
    """
    Транскрибирует аудио и сохраняет в файл.

    Args:
        audio_path (str): Путь к аудиофайлу
        save_path (str): Путь для сохранения транскрипции (по умолчанию "transcription.txt")

    Returns:
        str: Текст транскрипции
    """
    result = model.transcribe(audio_path)
    transcribed_text = result['text']

    with open(save_path, "w", encoding="utf-8") as f:
        f.write(transcribed_text)

    return transcribed_text

# Суммаризация

In [ ]:
from google.colab import userdata
import re

In [ ]:
# LLM API

import openai

# Настройки Yandex Cloud для работы с LLM
YANDEX_CLOUD_FOLDER = "" # вставьте id папки из консоли Yandex.Cloud
YANDEX_CLOUD_API_KEY = userdata.get('YANDEX_API') # безопасно извлекаем ключ (сохраните его в Secrets)
YANDEX_CLOUD_MODEL = "aliceai-llm/latest" # имя модели

# Инициализация клиента Yandex Cloud API
client = openai.OpenAI(
  api_key=YANDEX_CLOUD_API_KEY,
  base_url="https://ai.api.cloud.yandex.net/v1",
  project=YANDEX_CLOUD_FOLDER
)

In [ ]:
# Промпт для LLM: как суммаризировать лекцию и создать задания

INSTRUCTIONS = """
Суммаризируй транскрипт.
Ты методист и редактор учебных материалов. Твоя задача — создать структурированное, детальное саммари лекции на основе её транскрипта.
Саммари должно быть кратким, но содержательным. В то же время результат должен подаваться как самостоятельный текстовый урок, будто бы и не было никакого транскрипта другой лекции. Следуй инструкциям точно и строго.

ШАГ 1 - ФИЛЬТРАЦИЯ

Перед обработкой прочти транскрипт и исключи эти фрагменты:

- Просьбы написать в чат, поставить реакцию, лайк, плюсик, поднять руку, отметиться
- Технические ремарки: включить / выключить микрофон или камеру, проблемы со связью, зависания
- Упоминания имён участников, реплики о том, кто что написал в чате, кто включил камеру, чьи комментарии зачитываются вслух
- Периоды практической работы в сессионных залах (breakout rooms), когда участники молча выполняют задание: уходы в группы, возвраты, короткие дебрифы без содержательного разбора
- Любые фрагменты, не несущие новой методической или образовательной информации
- Повторения одинаковых по смыслу фрагментов

ШАГ 2 - СТРУКТУРА САММАРИ

Создай саммари строго в следующем формате. Используй именно эти заголовки и порядок разделов.

Саммари лекции "[Название лекции]"

Основная информация

Тема:
Сформулируй точное название темы лекции — 2-3 предложения. Если в транскрипте есть официальное название, то используй его. Добавь краткое пояснение (2-3 предложения), о чём лекция, если название краткое или очень общее.

Целевая аудитория лекции:
Укажи, для кого предназначена лекция: профессиональная роль, уровень подготовки, контекст.

Цель лекции:
Что ученик должен узнать, понять или начать делать по итогам лекции. Формулируй конкретно, 1–3 предложения.

Ключевые понятия (глоссарий):

Перечисли все значимые термины, техники, концепции и аббревиатуры, которые встречались в лекции.

Формат каждого пункта:
Термин — определение, данное лектором или вытекающее из контекста.

Включай только термины, которые лектор объяснял или на которые делал акцент. Не включай общеизвестные слова. Не выдумывай и не добавляй от себя.

---
Основное содержание лекции:

Напиши 1 абзац (3–5 предложений) — сжатое изложение всей лекции. Что было главным, о чём шла речь в целом.

Затем разбей содержание на блоки по логическим смысловым фрагментам

Для каждого блока:

Блок N

[Название блока. Коротко, отражает суть]

Общая суть
3–4 предложения: о чём этот блок, какова его функция в лекции.

Ключевые тезисы:
Список из 3–7 пунктов. Каждый пункт — это полноценная, развёрнутая мысль-предложение (не просто слово или фраза). Сохраняй конкретику: цифры, примеры, названия инструментов, условия, примечания. Пиши своими словами, но точно передавай смысл. Не придумывай от себя.

Блок N

[Продолжай по той же схеме для каждого смыслового блока лекции]
---

Выводы и итоги

Сформулируй 4–8 ключевых выводов лекции. Каждый отдельным пунктом, конкретным кратким тезисом, а не общей фразой. Это должно быть то, что участник вынесет из лекции как самое важное.

ШАГ 3 - ЗАДАНИЯ

На основе готового саммари создай задания пяти типов строго в указанном порядке.

Общие правила для всех заданий:
- Все вопросы опираются только на содержание саммари. Не добавляй информацию
  извне и не выдумывай
- Охватывай разные блоки саммари, не концентрируй все задания на одном блоке
- Не дублируй смысл вопросов внутри одного типа и между типами
- Используй термины и формулировки из саммари
- Язык заданий тот же, на котором написано саммари


ТИП 1. Тест с выбором одного ответа
Количество: 3–4 вопроса.

Проверить точное понимание терминов, определений, последовательностей и фактов — включая нюансы формулировок.

Требования:
- Один правильный ответ из 3–4 вариантов
- Все варианты правдоподобны, неправильные не должны быть очевидно абсурдными; Неправильные варианты — это намеренно близкие к правильному формулировки: та же идея, но с подменой термина, изменённым условием, нарушенной последовательностью или смещённым акцентом.
- Правильный ответ не должен быть длиннее остальных. Все варианты одинаковой длины. Если один вариант короче, дополни его уточняющим словом или контекстом, не меняя смысл. Нельзя выравнивать длину за счёт “воды” или бессмысленных добавлений
- Исключи ситуации, в которых правильный ответ находится всегда под одной и той же буквой (разные буквы: a, b, c, d не должны повторяться как правильный ответ более одного раза в рамках вопросов ТИП 1)


Формат вывода:

Вопрос [N]
[Текст вопроса]
a) [вариант]
b) [вариант]
c) [вариант]
d) [вариант]
✓ Правильный ответ: [буква] — [1–2 предложения: почему именно этот вариант верный]


ТИП 2. Распределение по категориям
Количество: 1-2 задания

Назначение: проверить умение классифицировать понятия, техники или ситуации по категориям из лекции — не просто знать термин, а понимать, к какой группе он относится и почему.


Требования:
Задание содержит 3 чётко названные категории из саммари (термины, этапы, типы, характеристики и т.п.)
Для распределения предлагается 3-6 элементов: понятий, примеров, утверждений или действий
Каждый элемент однозначно принадлежит одной категории — не допускай спорных или двусмысленных случаев
Элементы формулируются на языке саммари, без подсказок
Эталонный ответ содержит полное распределение с кратким пояснением логики


Формат вывода:

Задание [N]
Распределите перечисленные элементы по категориям: [Категория A], [Категория B], [Категория C].

Элементы для распределения:
1. [элемент]  2. [элемент]  3. [элемент]

✓ Правильное распределение:
[Категория A]: [номер элемента]
[Категория B]: [номер элемента]
[Категория C]: [номер элемента]



ТИП 3. Верно / Неверно с обязательным объяснением
Количество: 2–3 утверждения.

Назначение: научить студента видеть границы понятий, отличать точную
формулировку от правдоподобной, но неверной.

Требования:
- Каждое утверждение звучит правдоподобно, но содержит фактическую
  неточность, подмену понятия или нарушение логики из лекции
- Не используй очевидно абсурдные утверждения
- Ответ «верно» или «неверно» без объяснения не засчитывается — это должно быть указано в формулировке задания
- Чередуй верные и неверные утверждения — все утверждения не должны относиться к одной группе

Формат вывода:

Утверждение [N]
[Текст утверждения]
Верно или неверно? Объясните свой ответ в 1-2 предложениях.
✓ Ответ: [Верно / Неверно] — [1-2 предложения: в чём точность или
в чём именно ошибка и как правильно]

ТИП 4 — Мини-кейс
Количество: 1–2 кейса.

Назначение: проверить умение применить конкретную технику или концепцию
из лекции к реальной ситуации.

Требования:
- Кейс —  это короткая ситуация (3–5 предложений) с описанием контекста,
  участников и проблемы или момента, требующего действия
- Ситуация должна быть реалистичной и напрямую связанной с материалом лекции
- Вопрос к кейсу требует пошагового или обоснованного ответа — не «что это такое», а «что вы сделаете и почему»
- Эталонный ответ содержит опорные точки (не единственно верный сценарий,
  а ключевые элементы, которые должны присутствовать в ответе студента)

Формат вывода:

Кейс [N]
[Описание ситуации]
[Вопрос: что вы сделаете / как поступите / опишите свои действия]
✓ Опорные точки ответа: [перечисли 3–5 ключевых элементов,
которые должны быть в ответе студента]

ТИП 5 — Открытый вопрос с опорой на личную практику
Количество: 1–2 вопроса.

Назначение: перевести знание из абстрактного в личное  — студент
соотносит материал лекции со своим реальным опытом и формулирует позицию.

Требования:
- Вопрос не имеет единственно правильного ответа
- Обязательно содержит указание опереться на личный опыт или
  привести пример из своей практики
- Требует развёрнутого ответа — 4–6 предложений
- Начинается с конструкций: «Вспомните из своей практики...»,
  «Как вы считаете, опираясь на свой опыт...», «Был ли в вашей
  работе случай, когда...», «Как бы вы применили... в своей практике»

Формат вывода:

Вопрос [N]
[Текст вопроса]
Подсказка: [1 предложение — на что опереться при ответе,
без подсказки самого ответа]

ИТОГОВАЯ СТРУКТУРА ВЫВОДА

После выполнения всех шагов документ должен содержать разделы
строго в таком порядке:

1. Саммари лекции
   - Основная информация (тема, аудитория, цель)
   - Ключевые понятия (глоссарий)
   - Основное содержание по блокам
   - Выводы и итоги

2. Задания
   - Тип 1: Тест с выбором одного ответа
   - Тип 2: Распределение по категориям
   - Тип 3: Верно / Неверно с объяснением
   - Тип 4: Мини-кейс
   - Тип 5: Открытый вопрос с опорой на личную практику



ТРЕБОВАНИЯ К КАЧЕСТВУ:

- Язык саммари тот же, на котором велась лекция.
- Стиль академический, но доступный для понимания.
- Не используй прошедшее время, говоря о содержании лекции, используй настоящее или будущее. Вместо “в лекции рассказывалось” пиши “в лекции рассказывается” и т.п.
- Не упоминай лектора, не говори “лектор сказал, показал” и т.д. Помни, что данная саммаризация является самостоятельным независимым уроком
- Терминология — сохраняй формулировки лектора, не подменяй смысл синонимами и не заменяй на выдуманные термины.
- Конкретность — если лектор называл цифры, инструменты, условия, исследования, включай их.
- Объём блоков достаточен для понимания его сути.
- Структура блоков — отражает реальную логику лекции, а не произвольную или исключительно хронологическую разбивку.

Перед выводом перепроверь себя: у тебя должен получиться не дословный пересказ по заданной структуре, а облегченное и при этом содержательное саммари, которое можно использовать в качестве письменного учебного материала для быстрого изучения материалов лекции.
"""

In [ ]:
def summarize(message):
    """
    Преобразует расшифровку вебинара в структурированный урок.

    Args:
        message (str): Текст расшифровки из Whisper

    Returns:
        str: Структурированный урок с темой, содержанием, выводами и заданиями
    """
    response = client.responses.create(
        model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
        temperature=0.3,
        instructions=INSTRUCTIONS,
        input=message,
        max_output_tokens=20000
        )
    return response.output_text

In [ ]:
def create_files(summary, save_path_md="summary.md", save_path_txt="summary.txt"):
    """
    Сохраняет суммаризацию в MD и TXT файлы.

    MD файл сохраняет оригинальный Markdown с форматированием.
    TXT файл сохраняет очищенный текст без Markdown-разметки.

    Args:
        summary (str): Текст суммаризации для сохранения
        save_path_md (str): Путь для сохранения MD файла (по умолчанию "summary.md")
        save_path_txt (str): Путь для сохранения TXT файла (по умолчанию "summary.txt")

    Returns:
        tuple: (путь_к_md, путь_к_txt) - кортеж из двух строк
    """
    with open(save_path_md, "w", encoding="utf-8") as f:
        f.write(summary)

    with open(save_path_txt, "w", encoding="utf-8") as f:

        clean_text = summary
        clean_text = re.sub(r'\*\*(.*?)\*\*', r'\1', clean_text)
        clean_text = re.sub(r'\*(.*?)\*', r'\1', clean_text)
        clean_text = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', clean_text)
        clean_text = re.sub(r'`(.*?)`', r'\1', clean_text)
        clean_text = re.sub(r'#{2,}\s*', ' ', clean_text)
        f.write(clean_text)

    return save_path_md, save_path_txt

# Запуск бота

In [ ]:
!pip install pyTelegramBotAPI openai -q

In [ ]:
import telebot
import os

In [ ]:
# Telegram Bot
# Скачивает файлы с Яндекс.Диска, транскрибирует через Whisper,
# суммаризирует через Yandex Cloud LLM, отправляет результат пользователю

# тг-бот
bot = telebot.TeleBot(userdata.get('TG_BOT')) # безопасно извлекаем API-ключ для бота, сохраните в Secrets

@bot.message_handler(commands=['start'])
def start(message):
    # приветствие
    welcome_text = """
🎓 Привет! Я YaSumBot, бот для суммаризации вебинаров.

Устали переслушивать часы лекций? Я превращаю их в понятный конспект за минуты!

✨ Что я делаю:
• Принимаю ссылку на аудио/видео с Яндекс.Диска
• Превращаю речь в печатный текст
• Выжимаю самое важное
• Добавляю задания с ответами по пройденному материалу

📂 Я поддерживаю любые форматы аудио и видео с Яндекс.Диска

⚠️ Важные ограничения:
• Рекомендуемая длительность: до 3 часов
• Только публичные ссылки Яндекс.Диска
• Не копируйте ссылку из адресной строки, используйте иконку "Поделиться" > "Скопировать"
• Отправляйте ссылку на файл, а не на папку
• Обработка занимает ~15 минут для вебинара длительностью 1.5 часа

📚 Что вы получите:
1. Полную расшифровку вебинара (TXT)
2. Краткое содержание с заданиями (Markdown и TXT)

💡 Подходит для:
• Лекций и семинаров
• Вебинаров и конференций
• Подкастов и интервью
___________________

🔗 Отправьте публичную ссылку на файл с Яндекс.Диска, и я начну работу!
"""

    bot.reply_to(message, welcome_text)

@bot.message_handler(content_types=['text'])
def handle_text(message):
    """Обрабатывает ссылки, скачивает файл, транскрибирует и суммаризирует."""
    url = message.text.strip()

    # Проверка на ссылку
    if not url.startswith(('http://', 'https://')):
        bot.reply_to(
            message,
            "Очень интересно, но это не ссылка!\n\n"
            "🔗 Отправьте ссылку, начинающуюся с http:// или https://"
            )
        return

    # Проверка на корректность ссылки с Яндекс.Диска
    is_valid, msg, filename = is_yandex_file(url)

    if not is_valid:
        bot.reply_to(message, f"😢 {msg}")
        return

    bot.reply_to(message, f"{msg}")

    try:
        # Скачивание файла
        bot.reply_to(message, f"🚀 Скачиваю файл...")

        temp_audio = "temp_audio.ogg"

        if download_yandex_file(url, temp_audio):
            # Транскрибация аудио в текст
            bot.reply_to(
                message,
                "🎤 Распознаю текст... Это может занять несколько минут"
                )

            transcribed_text = transcribe(temp_audio, "transcription.txt")

            # Отправка расшифровки
            with open("transcription.txt", "rb") as f:
                bot.send_document(
                    message.chat.id,
                    f,
                    caption="📚 Полная расшифровка лекции!"
                )

             # Суммаризация текста
            bot.reply_to(message, "✍️ Суммаризирую текст...")
            summary = summarize(transcribed_text)
            create_files(summary, "summary.md", "summary.txt")

            # Отправка саммари в формате MD
            with open("summary.md", "rb") as f:
                bot.send_document(
                    message.chat.id,
                    f,
                    caption="📄 Готовый суммаризированный урок в Markdown!"
                )

            # Отправка саммари в формате TXT
            with open("summary.txt", "rb") as f:
                bot.send_document(
                    message.chat.id,
                    f,
                    caption="📄 Готовый суммаризированный урок в TXT!"
                )

            # Очистка временных файлов
            for file in [
                "summary.md",
                "summary.txt",
                "transcription.txt",
                temp_audio
                ]:
                if os.path.exists(file):
                    os.remove(file)

        else:
            bot.reply_to(
                message,
                "☹️ Не удалось скачать файл. Проверьте ссылку"
                )

    # Обработка непредвиденных ошибок
    except Exception as e:
        bot.reply_to(
            message,
            "😟 Ой! Что-то пошло не так!"
            "\nПопробуйте ещё раз или отправьте другой файл")

bot.infinity_polling()

2026-06-08 16:29:47,181 (__init__.py:1147 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
ERROR:TeleBot:Infinity polling: polling exited
2026-06-08 16:29:47,184 (__init__.py:1149 MainThread) ERROR - TeleBot: "Break infinity polling"
ERROR:TeleBot:Break infinity polling
